# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaminari19/FlyRank-Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My answer:** One row = **one `(client_hash_id, content_hash_id)` pair, for one month** —
built by aggregating `fact_content_daily_performance`'s daily rows (whose own grain is
`report_date × client_hash_id × content_hash_id`) up to the content-month level. Features come
from **`month = 2026-03`**; the label compares that to **`month = 2026-02`**. Both are mid-panel
partitions, chosen deliberately over `fact_content_daily_performance_sample` (June 2026, the
panel's sealed final month) so I'm not developing label logic inside my own eventual test window.

In [21]:
%pip install -q duckdb huggingface_hub

import duckdb
from huggingface_hub import HfApi, login

# Colab: put your HF token in the Secrets (key icon) panel as HF_TOKEN — never paste it in a cell,
# this repo is public.
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

REPO_ID = "FlyRank/internship-warehouse"

# --- Discovery step: confirm the real file/partition layout before guessing paths ---
api = HfApi()
files = api.list_repo_files(REPO_ID, repo_type="dataset")
for f in sorted(files):
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [22]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}');")

# Mid-panel month, per skills/flyrank/flyrank-data: iterate here, never on the _sample
# (_sample = June 2026, the panel's sealed final month / natural test window).
MONTH = "2026-03"
PREV_MONTH = "2026-02"

# ADJUST if the file listing above shows a different naming pattern
FACT_MONTH  = f"hf://datasets/{REPO_ID}/fact_content_daily_performance/month={MONTH}/*.parquet"
FACT_PREV   = f"hf://datasets/{REPO_ID}/fact_content_daily_performance/month={PREV_MONTH}/*.parquet"
DIM_CLIENTS = f"hf://datasets/{REPO_ID}/dim_clients.parquet"
DIM_CONTENT = f"hf://datasets/{REPO_ID}/dim_content.parquet"
# fact_content_query_90d and fact_content_daily_performance_sample deliberately NOT loaded — see Section 2.

# Cheap smoke test: read ONLY the one partition, not the full 78.8M-row table
con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{FACT_MONTH}')").show()

┌─────────┐
│    n    │
│  int64  │
├─────────┤
│ 9841378 │
└─────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**My answer:**

- **Tables used:** `fact_content_daily_performance`, at the `month=2026-03` and `month=2026-02`
  partitions only (never a full-table scan), joined to `dim_content` for static attributes.
- **Features** (from `month=2026-03`, GA4 fields filtered on availability — see 3c):
  `clicks_month`, `impressions_month`, `avg_position_month`, `engaged_sessions_month`,
  `content_age_days`.
- **Label/proxy:** `is_declining` = 1 if `clicks_month` (March) < `clicks_prev_month` (February),
  both computed from the **same table**, at aligned calendar-month windows. A defined-rule proxy
  on an observed click trend — not a true "needs refresh" outcome. `clicks_prev_month` itself is
  held **out** of the feature set (see the trap in 3e).
- **Context (not fed to a model):** `client_hash_id`, `content_hash_id`, `content_type`,
  `is_published`, `is_deleted` — identity/grouping/filtering only.
- **Excluded, with why:**
  - `fact_content_query_90d` — entirely excluded. Its 90-day window is **fixed**, not partitioned
    by month, and overlaps the panel's final months, so aligning it to a March 2026 feature
    window isn't straightforward. It also repeats per-content columns
    (`content_total_impressions_90d`, `content_visible_query_count`) on every query row —
    they need `ANY_VALUE()`, never `SUM()`, an easy silent double-count.
  - `fact_content_daily_performance_sample` — the panel's sealed final month; touching it near
    label logic means developing inside the future test window.
  - `search_volume`, `cpc`, `competition` (from `dim_content`) — fixed at keyword creation, not
    a monthly performance signal, and a different grain than my content-month unit.
- **Output:** a ranked list of `(client_hash_id, content_hash_id)` pairs for March 2026 with a
  refresh-priority score, so an editor works the top of the list first.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Grain check
One row should be one `(report_date, client_hash_id, content_hash_id)` — grouping by that triple
and keeping groups with `COUNT(*) > 1` should return **zero rows** if the grain claim holds.

### 3b. Row count + date span
How big is this partition, and does it actually span the calendar month I claimed?

### 3c. Availability — filtered with `IS TRUE`
How many rows have usable GSC *and* GA4 data, before vs. after filtering? GA4 columns are
zero-filled (not genuinely zero) before a client's `ga4_data_start`, so `ga4_data_available` has
to be checked explicitly, not inferred from the numbers themselves.

### 3d. Five features (max) — one content-month feature frame
Every feature gets one line: knowable at the decision moment because…


### 3e. The trap — one deliberate label-derived column
Build the label from March vs. February clicks (same table, aligned windows). Get an honest
score with the 5 features above — `clicks_prev_month` is deliberately withheld. Then add it back
in as the trap, watch the score jump toward perfect (the label is *defined* from it), delete it,
keep the honest number.

In [23]:
#3a

grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{FACT_MONTH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Duplicate-grain rows found: {len(grain_check)}  (expect 0)")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate-grain rows found: 0  (expect 0)


,report_date,client_hash_id,content_hash_id,c


In [24]:
#3b

span_check = con.sql(f"""
    SELECT
        COUNT(*)                        AS row_count,
        COUNT(DISTINCT client_hash_id)   AS distinct_clients,
        COUNT(DISTINCT content_hash_id)  AS distinct_content,
        MIN(report_date)                 AS first_date,
        MAX(report_date)                 AS last_date
    FROM read_parquet('{FACT_MONTH}')
""").df()

span_check

,row_count,distinct_clients,distinct_content,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [25]:
#3c

before = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{FACT_MONTH}')").df()["n"][0]

after = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{FACT_MONTH}')
    WHERE gsc_data_available IS TRUE
      AND client_has_gsc     IS TRUE
      AND ga4_data_available IS TRUE
      AND client_has_ga4     IS TRUE
""").df()["n"][0]

print(f"Before availability filter: {before:,} rows")
print(f"After  availability filter: {after:,} rows  ({after/before:.1%} survive)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Before availability filter: 9,841,378 rows
After  availability filter: 364,347 rows  (3.7% survive)


In [26]:
#3d

features_df = con.sql(f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks)            AS clicks_month,
            SUM(gsc_impressions)       AS impressions_month,
            AVG(gsc_avg_position)      AS avg_position_month,
            SUM(ga4_engaged_sessions)  AS engaged_sessions_month
        FROM read_parquet('{FACT_MONTH}')
        WHERE gsc_data_available IS TRUE AND client_has_gsc IS TRUE
          AND ga4_data_available IS TRUE AND client_has_ga4 IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        m.*,
        DATE_DIFF('day', d.content_created_date, DATE '{MONTH}-01' + INTERVAL 1 MONTH - INTERVAL 1 DAY) AS content_age_days
    FROM monthly m
    JOIN read_parquet('{DIM_CONTENT}') d USING (content_hash_id)
    WHERE d.is_published IS TRUE
      AND d.is_deleted IS NOT TRUE
""").df()

print(features_df.shape)
features_df.head()

# 1. clicks_month           — observed clicks through March month-end, no future data
# 2. impressions_month      — observed impressions through March month-end
# 3. avg_position_month     — observed ranking position during the window itself
# 4. engaged_sessions_month — observed GA4 engagement, only where ga4_data_available IS TRUE
# 5. content_age_days       — content_created_date is a fixed historical fact, always known

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(63847, 7)


,client_hash_id,content_hash_id,clicks_month,impressions_month,avg_position_month,engaged_sessions_month,content_age_days
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2.0,458.0,4.418032,0.0,274
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,23.0,3943.0,4.392897,1.0,274
2,client_65de48885f4ef01b,content_3c286ded8bd68120,15.0,2180.0,8.439390,2.0,265
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,8.0,503.0,5.531459,1.0,263
4,client_65de48885f4ef01b,content_ff867882e604fa96,0.0,24.0,2.850000,0.0,259


In [27]:
#3e

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

prev_month_clicks = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_prev_month
    FROM read_parquet('{FACT_PREV}')
    WHERE gsc_data_available IS TRUE AND client_has_gsc IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

data = features_df.merge(prev_month_clicks, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["clicks_month"] < data["clicks_prev_month"]).astype(int)
print("Rows with both March features and a February comparison:", len(data))
print(data["is_declining"].value_counts(normalize=True))

honest_features = ["clicks_month", "impressions_month", "avg_position_month",
                    "engaged_sessions_month", "content_age_days"]

def quick_auc(cols):
    X = data[cols].fillna(0)
    y = data["is_declining"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    return roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

honest_auc = quick_auc(honest_features)
print(f"Honest AUC (5 real features, clicks_prev_month withheld): {honest_auc:.3f}")

# --- THE TRAP: clicks_prev_month is literally half of how the label is defined ---
leaky_auc = quick_auc(honest_features + ["clicks_prev_month"])
print(f"Leaky AUC  (+ clicks_prev_month):  {leaky_auc:.3f}  <-- jumps toward 1.0, for the wrong reason")

# Delete it. Keep the honest number.
data = data.drop(columns=["clicks_prev_month"])
print(f"\nKeeping the honest AUC: {honest_auc:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with both March features and a February comparison: 51848
is_declining
0    0.62012
1    0.37988
Name: proportion, dtype: float64
Honest AUC (5 real features, clicks_prev_month withheld): 0.619
Leaky AUC  (+ clicks_prev_month):  1.000  <-- jumps toward 1.0, for the wrong reason

Keeping the honest AUC: 0.619


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**My answer (named limitation):** Rows before a client's `ga4_data_start` are zero-filled for GA4
columns, flagged by `ga4_data_available = FALSE`. My 3c filter drops those rows correctly, but that
means clients who connected GA4 later in the panel are structurally **under-represented** in this
slice, not missing at random — a client with a late `ga4_data_start` contributes fewer content-month
rows than one tracked from day one, even if their actual content performance is similar. A single
global calendar window (March 2026) also can't distinguish "this client had no content that month"
from "this client's `gsc_data_start` was after March" — `dim_clients.gsc_data_start` would need to
be checked per client to tell those apart, which this contract doesn't yet do.

## Self-check

Before you submit, confirm each line honestly:

- [/] Every section above is filled — markdown thinking AND the code that backs it
- [/] The notebook runs top to bottom with no errors (Runtime → Run all)
- [/] No client names, URLs, or private queries anywhere
- [/] My claims use careful words: observed, measured, directional, decision-support
- [/] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.